# L1 -> L2

In [ ]:
from romancal.pipeline import ExposurePipeline as ELP
import os
import warnings
import logging
from glob import glob as gb
    
# Ignore warnings
warnings.filterwarnings('ignore')
# Ignore INFO
logging.getLogger().setLevel(logging.CRITICAL)

# Reduce uncal files
ufiles = gb('MAST_2026-10-09T1317/Roman/*_uncal.asdf')
steps = {'source_catalog':{'skip': True}}
for ufile in ufiles:
    res = ELP.call(ufile, steps=steps, save_results=True)

# Background files

In [ ]:
rom glob import glob as gb
import os
from flo.wfiskymatch import asdf2healpix
path = '/Users/dfadda/Roman/FLO_Test/Orion/'
outdir = path + 'Bkg/'
# Check if directory exists, otherwise creates it
os.makedirs(outdir, exist_ok=True)
files = sorted(gb(os.path.join(path,'*_cal.asdf')))
for file in files:
    asdf2healpix(file, outdir, savefits=False)  # NB: if True, background files are also saved as FITS files

# Offsets computation

In [ ]:
# Computing offsets
from flo.wfiskymatch import pixmatch
from glob import glob as gb
import os

path = '/Users/dfadda/Roman/FLO_Test/Orion/'
outdir = path + 'Bkg/'
files = sorted(gb(os.path.join(outdir,'*cal.h5')))

object = 'field'
offsets = pixmatch(files, outfile=outdir+object+'_offsets', offsets=None)

for file, offset in zip(files, offsets):
    print(file, offsets)

# Offset files

Adding computed offsets to original cal files

In [ ]:
from flo.wfiskymatch import addOffset
import h5py
import os

path = '/Users/dfadda/Roman/FLO_Test/Orion/'
bkgpath = path + 'Bkg/'
h5file = bkgpath + 'field_offsets.h5'
with h5py.File(h5file, 'r') as hdf5_file:
    files = hdf5_file['files'][:]
    offsets = hdf5_file['offsets'][:]

# Create new files with added offsets
outdir = path + '/Offset/'
os.makedirs(outdir, exist_ok=True)
for file, offset in zip(files,offsets):
    print(file.decode(), offset)
    L2 = path + file.decode() +'.asdf'
    addOffset(L2, offset=offset, outdir = outdir)

# Desaturate stars

## Obtaining the GAIA catalog

This passage requires the availability of the GAIA catalog for the field.
Once acquired and saved a catalog covering the entire area, this can be used for all the filters.

In [ ]:
# Get the list of coordinates at the time of the observation
from flo.simutils import gaiastars
import roman_datamodels as rdm

# Use a central image to get the position or just write the position of the survey center
with rdm.open('Offset/r0108303005001001002_0002_wfi16_f213_off_cal.asdf') as dm:
    astro = dm.meta.wcs
    ra0, dec0 = astro(2044.5,2044.5)
    start_time = dm.meta.exposure.start_time

# Radius of the catalog in degrees
radius = 0.5  

# Retrieve catalog and save it locally
gaia_catalog = gaiastars(ra=ra0, dec=dec0, radius=radius, star=0.5, maglim=16, obs_time=start_time)
gaia_catalog.write('gaia_catalog.ecsv', format='ascii.ecsv', overwrite=True)

## Desaturate the stars in the survey

In [ ]:
# Desaturate the PSFs
from flo.desaturate import desaturatePsf
from astropy.coordinates import SkyCoord
from astropy.table import Table
from glob import glob as gb
import os

path = '/Users/dfadda/Roman/FLO_Test/Orion/Offset/'
ufiles = gb(path + 'r*_off_cal.asdf')

gaia_catalog = Table.read('gaia_catalog.ecsv')
stars = SkyCoord(gaia_catalog['ra'].value, gaia_catalog['dec'].value, unit='deg', frame = 'icrs')

for L2 in ufiles:
    desaturatePsf(L2, stars)
    # Remove the offset file
    os.remove(L2)

# Masking of bad pixels

Here we should also include possible masking of regions affected by artefacts.
Updated masks will be generated asap.

In [ ]:
from glob import glob as gb
from flo.badpixels import maskNegativPixels as maskNP
import numpy as np
import os 

path = '/Users/dfadda/Roman/FLO_Test/Orion/Offset/'
ufiles = gb(path + 'r*_off_cal_us.asdf')

for ufile in ufiles:
    maskNP(ufile)
    os.remove(ufile)

# Mosaic

There are two ways to obtain a mosaic:

1. direct, using all the images at once

2. tiled, creating tiles and then stitching them together

## Direct way

### Create the association file

In [ ]:
# Association file
import romancal.associations.asn_from_list as asn_from_list
from glob import glob as gb

path = '/Users/dfadda/Roman/FLO_Test/Orion/Offset/'
input_files = gb(path + 'r*_ok.asdf')
product_name = 'Orion_f213'

# Create the ASN JSON object
asn = asn_from_list.asn_from_list([(i, 'science') for i in input_files], 
                                  product_name=product_name, 
                                  with_exptype=True, target='')

# Write the ASN file to disk
_, serialized = asn.dump()
with open(path+'Orion_f213_asn.json', 'w') as f:
    f.write(serialized)

### Coadd the images

In [ ]:
# Mosaic without computing new offsets
from romancal.pipeline import MosaicPipeline

path = '/Users/dfadda/Roman/FLO_Test/Orion/Offset/'

steps = {'outlier_detection':{'skip':True},
         'skymatch':{'skip':True},
         'resample':{'resample_on_skycell':False},
         'source_catalog':{'skip':True},
        }

result = MosaicPipeline.call(path+'Orion_f213_asn.json', 
                             save_results=True, resample_on_skycell=False,
                             steps=steps)

### Display the result

In [ ]:
# Display mosaic
import matplotlib.pyplot as plt
import roman_datamodels as rdm
from astropy import visualization
from astropy.table import Table
import numpy as np
from astropy.wcs.utils import proj_plane_pixel_scales as ppps

filename = 'Orion_f213_coadd.asdf'

with rdm.open(filename) as dm:
    wcs = dm.meta.wcs
    meta = dm.meta
    data = dm.data.copy()
    hdr = dm.meta.wcs.to_fits()[0]
    print('Pixel scale ', dm.meta.resample['pixel_scale_ratio'],' pixels')

from astropy.wcs import WCS
wcs = WCS(hdr)
print('Pixel scale from wcs ', ppps(wcs)*3600, ' arcsec')
print('size of skycell: ', np.shape(data))

norm = visualization.ImageNormalize(data,
                                    interval=visualization.ZScaleInterval(contrast=1),
                                    stretch=visualization.AsinhStretch(a=1))

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'projection':wcs})

plot = ax.imshow(data, norm=norm, origin='lower',cmap='inferno_r')
ax.coords.grid(True, color='white', ls='dashed')
ax.coords[0].set_axislabel('R.A. (J2000)')
ax.coords[1].set_axislabel('Dec. (J2000)')

cax = plt.axes([0.925, 0.11, 0.025,0.77 ])
plt.colorbar(mappable=plot, cax=cax)

plt.show()

### Save as a FITS file

In [ ]:
# Original mosaic to FITS
import roman_datamodels as rdm
from astropy.io import fits

filename = 'Orion_f213_coadd.asdf'

with rdm.open(filename) as dm:
    data = dm.data.copy()
    hdr = dm.meta.wcs.to_fits()[0]

fits.writeto('Orion_f213_coadd.fits', data, hdr, overwrite=True)

## Via tiling

Before running the mosaicker, one has to change the vparity in romancal.

In your romancal repository, open the file:
```
romancal/skycell/skymap.py 
```
and change the line after:
```
matrix = wcsinfo.get("rotation_matrix", None)
```
to: 
```
vparity=-1,
```
The next version of romancal will have vparity=-1 by default.

### Create the association files

In [ ]:
from flo.mosaic import make_cells_asn
from glob import glob as gb
import os

path = '/Users/dfadda/Roman/FLO_Test/Orion/Offset/'
files = gb(path + 'r*_ok.asdf')
refdir = os.environ.get('CRDS_PATH')+'references/roman/wfi/'
outname = 'tile'
make_cells_asn(files, outname=outname, refdir=refdir)

### Obtain the tiles

In [ ]:
# Tiles 
from glob import glob as gb
from romancal.pipeline import MosaicPipeline
from flo.skymap import _wcsinfo_to_wcs

path = '/Users/dfadda/Roman/FLO_Test/Orion/'
outname = 'tile'
asnfiles = gb(path+outname+'*.asn')

steps = {'outlier_detection':{'skip':True},
         'skymatch':{'skip':True},
         'resample':{'resample_on_skycell':True},
         'source_catalog':{'skip':True},
        }

for asnfile in asnfiles:
    print('asnfile is: ', asnfile)
    result = MosaicPipeline.call(asnfile, 
                                 save_results=True, resample_on_skycell=True,
                                 steps=steps)

### Stitch the tiles

In [ ]:
from glob import glob as gb
from flo.mosaic import stitchCells

path = '/Users/dfadda/Roman/FLO_Test/Orion/'
outname = 'tile'
asdffiles = gb(path+outname+'*.asdf')
stitchCells(asdffiles, outname='Orion_tiles')

### Display the mosaic with tiles

In [ ]:
# Display mosaic
import matplotlib.pyplot as plt
import roman_datamodels as rdm
from astropy import visualization
from astropy.table import Table
import numpy as np
from astropy.wcs.utils import proj_plane_pixel_scales as ppps
from astropy.wcs import WCS
from astropy.io import fits

with fits.open('Orion_tiles.fits') as f:
    image = f[0].data
    header = f[0].header

wcs = WCS(header)

for k in header.keys():
    print(k, header[k])

print(wcs)

#wcs = WCS(hdr)
print('Pixel scale from wcs ', ppps(wcs)*3600, ' arcsec')
print('size of image: ', np.shape(image))


norm = visualization.ImageNormalize(image,
                                    interval=visualization.ZScaleInterval(contrast=1),
                                    stretch=visualization.AsinhStretch(a=1))

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'projection':wcs})

plot = ax.imshow(image, norm=norm,  origin='lower',cmap='inferno_r')
ax.coords.grid(True, color='white', ls='dashed')
ax.coords[0].set_axislabel('R.A. (J2000)')
ax.coords[1].set_axislabel('Dec. (J2000)')

skyref = '/Users/dfadda/data/crds_cache/references/roman/wfi/roman_wfi_skycells_0001.asdf'
with rdm.open(skyref) as dm:
    tiles = dm.projection_regions.copy()
    skycells = dm.skycells.copy()
for skycell in skycells:
    rac = [skycell['ra_corn1'],skycell['ra_corn2'],skycell['ra_corn3'],skycell['ra_corn4'],skycell['ra_corn1']]
    dec = [skycell['dec_corn1'],skycell['dec_corn2'],skycell['dec_corn3'],skycell['dec_corn4'],skycell['dec_corn1']]
    plt.plot(rac, dec,color='black',transform=ax.get_transform('icrs'))
    plt.text(skycell['ra_center'], skycell['dec_center'], skycell['name'],transform=ax.get_transform('icrs'))
    print(skycell['name'],skycell['ra_center'], skycell['dec_center'])

cax = plt.axes([0.925, 0.11, 0.025,0.77 ])
plt.colorbar(mappable=plot, cax=cax)
plt.grid()
plt.show()